# Part 1: Load the data

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

train.head()
train.shape
train.info()
train.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


# Part2: machine learning model

## Data cleaning for the train.csv dataset

In [3]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
age_median = train['Age'].median()
fare_median = train['Fare'].median()
train['Age'] = train['Age'].fillna(age_median)
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
train = pd.get_dummies(train, columns=['Embarked'])

## Feature engineering

In [5]:
train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.')

train['Title'] = train['Title'].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 
    'Rare'
)
train['Title'] = train['Title'].replace(['Mlle', 'Mme', 'Ms'], 'Rare')
train['Title'] = train['Title'].map(
    {'Mr':0, 'Miss':1, 'Mrs':2, 'Master':3, 'Rare':4}
)
train['Title'] = train['Title'].fillna(4)

train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)

train['AgeBand'] = pd.cut(train['Age'], bins=[0,12,18,35,60,80], labels=[0,1,2,3,4])

train['HasCabin'] = train['Cabin'].notna().astype(int)

train['TicketPrefix'] = train['Ticket'].str.extract(r'([A-Za-z]+)', expand=False)
train['TicketPrefix'] = train['TicketPrefix'].fillna('NONE')
train['HasTicketPrefix'] = (train['TicketPrefix'] != 'NONE').astype(int)

In [6]:
print(train.columns.tolist())
train.head()

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title', 'FamilySize', 'IsAlone', 'AgeBand', 'HasCabin', 'TicketPrefix', 'HasTicketPrefix']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Embarked_C,Embarked_Q,Embarked_S,Title,FamilySize,IsAlone,AgeBand,HasCabin,TicketPrefix,HasTicketPrefix
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,...,False,False,True,0,2,0,2,0,A,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,...,True,False,False,2,2,0,3,1,PC,1
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,...,False,False,True,1,1,1,2,0,STON,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,...,False,False,True,2,2,0,2,1,NONE,0
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,...,False,False,True,0,1,1,2,0,NONE,0


## Build the random forest model

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'AgeBand',
            'Title', 'FamilySize', 'IsAlone', 'Embarked_C', 'Embarked_Q', 'Embarked_S',
            'HasCabin', 'HasTicketPrefix']

X = train[features]
y = train['Survived']


model = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_split=8, min_samples_leaf=4, max_features='sqrt',random_state=42)
model.fit(X, y)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=skf)
print(f"cross-validation score: {scores}")
print(f"average accuracy: {scores.mean():.4f}")
print(f"standard deviation: {scores.std():.4f}")

cross-validation score: [0.83240223 0.8258427  0.8258427  0.83146067 0.83707865]
average accuracy: 0.8305
standard deviation: 0.0043


# Part3: Prediction and Submission

## Apply the same processing (cleaning + feature engineering) to the test set

In [8]:
test.head()
test.shape
test.info()
test.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.1+ KB


,PassengerId,Pclass,Age,SibSp,Parch,Fare
count,418.000000,418.000000,332.000000,418.000000,418.000000,417.000000
mean,1100.500000,2.265550,30.272590,0.447368,0.392344,35.627188
std,120.810458,0.841838,14.181209,0.896760,0.981429,55.907576
min,892.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,996.250000,1.000000,21.000000,0.000000,0.000000,7.895800
50%,1100.500000,3.000000,27.000000,0.000000,0.000000,14.454200
75%,1204.750000,3.000000,39.000000,1.000000,0.000000,31.500000
max,1309.000000,3.000000,76.000000,8.000000,9.000000,512.329200


In [9]:
test.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [10]:
test['Age'] = test['Age'].fillna(age_median)
test['Fare'] = test['Fare'].fillna(fare_median)

test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})
test = pd.get_dummies(test, columns=['Embarked'])

In [11]:
test['Title'] = test['Name'].str.extract(r' ([A-Za-z]+)\.')

test['Title'] = test['Title'].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 
    'Rare'
)
test['Title'] = test['Title'].replace(['Mlle', 'Mme', 'Ms'], 'Rare')
test['Title'] = test['Title'].map(
    {'Mr':0, 'Miss':1, 'Mrs':2, 'Master':3, 'Rare':4}
)
test['Title'] = test['Title'].fillna(4)

test['FamilySize'] = test['SibSp'] + test['Parch'] + 1
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

test['HasCabin'] = test['Cabin'].notna().astype(int)

test['AgeBand'] = pd.cut(test['Age'], bins=[0,12,18,35,60,80], labels=[0,1,2,3,4])

test['TicketPrefix'] = test['Ticket'].str.extract(r'([A-Za-z]+)', expand=False)
test['TicketPrefix'] = test['TicketPrefix'].fillna('NONE')
test['HasTicketPrefix'] = (test['TicketPrefix'] != 'NONE').astype(int)

In [12]:
print(test.columns.tolist())
test.head()

['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title', 'FamilySize', 'IsAlone', 'HasCabin', 'AgeBand', 'TicketPrefix', 'HasTicketPrefix']


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_C,Embarked_Q,Embarked_S,Title,FamilySize,IsAlone,HasCabin,AgeBand,TicketPrefix,HasTicketPrefix
0,892,3,"Kelly, Mr. James",0,34.5,0,0,330911,7.8292,NaN,False,True,False,0,1,1,0,2,NONE,0
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",1,47.0,1,0,363272,7.0000,NaN,False,False,True,2,2,0,0,3,NONE,0
2,894,2,"Myles, Mr. Thomas Francis",0,62.0,0,0,240276,9.6875,NaN,False,True,False,0,1,1,0,4,NONE,0
3,895,3,"Wirz, Mr. Albert",0,27.0,0,0,315154,8.6625,NaN,False,False,True,0,1,1,0,2,NONE,0
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",1,22.0,1,1,3101298,12.2875,NaN,False,False,True,2,3,0,0,2,NONE,0


## Predict the test set and generate a submission file

In [13]:
X_test = test[features]
predictions = model.predict(X_test)


submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})
submission.to_csv('submission.csv', index=False)